In [ ]:
from pathlib import Path
import json
import re
import html


# ----------------------------
# Find Learning Lab repo root
# ----------------------------

def find_repo_root(start_path):
    start_path = Path(start_path).resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "mkdocs.yml").exists():
            return path

    raise FileNotFoundError("Could not find mkdocs.yml. Run this from inside the Learning Lab repo.")


ROOT = find_repo_root(Path.cwd())

DATASET_REPO = ROOT.parent / "cloud-datasets"
DATASET_DIR = DATASET_REPO / "records"

# If there is no records folder, search the whole cloud-datasets repo
if not DATASET_DIR.exists():
    DATASET_DIR = DATASET_REPO

OUT_FILE = ROOT / "docs" / "rosetta-stone" / "datasets.md"
JS_FILE = ROOT / "docs" / "javascripts" / "dataset-filter.js"

print("Learning Lab root:", ROOT)
print("Dataset JSON folder:", DATASET_DIR)
print("Output Markdown:", OUT_FILE)
print("Output JavaScript:", JS_FILE)


# ----------------------------
# Helpers
# ----------------------------

def esc(value):
    """Escape text for safe HTML display."""
    if value is None:
        return ""
    return html.escape(str(value), quote=True)


def safe_id(value):
    """Create a safe HTML id."""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", "-", value)
    return value.strip("-") or "dataset"


def doi_url(doi):
    """Convert DOI text into a DOI URL."""
    if not doi:
        return ""

    doi = str(doi).strip()

    if doi.startswith("http://") or doi.startswith("https://"):
        return doi

    return f"https://doi.org/{doi}"


def doi_link(doi):
    """Create a clickable DOI link."""
    if not doi:
        return "TBD"

    doi = str(doi).strip()
    url = doi_url(doi)
    label = doi.replace("https://doi.org/", "").replace("http://doi.org/", "")

    return f'<a href="{esc(url)}" target="_blank" rel="noopener">{esc(label)}</a>'


def version_key(version):
    """Sort versions like v2.0.3 above v2.0.2."""
    numbers = re.findall(r"\d+", str(version))

    if not numbers:
        return (0,)

    return tuple(int(number) for number in numbers)


def get_dataset_id(dataset):
    """Return dataset ID as a string, even if source JSON has an int."""
    dataset_id = (
        dataset.get("id")
        or dataset.get("name")
        or "unknown-dataset"
    )

    return str(dataset_id)


def get_dataset_title(dataset):
    return str(dataset.get("title") or dataset.get("name") or get_dataset_id(dataset))


def get_collection(dataset):
    """Return collection value or NA when missing/blank."""
    collection = dataset.get("collection")

    if collection is None:
        return "NA"

    if isinstance(collection, list):
        if not collection:
            return "NA"
        return ", ".join(str(item) for item in collection)

    if str(collection).strip() == "":
        return "NA"

    return str(collection)


def get_tags(dataset):
    """
    Return tags for filtering.
    Uses dataset['tags'] if present and also includes dataset['keywords'].
    """
    tags = dataset.get("tags", [])
    keywords = dataset.get("keywords", [])

    if not isinstance(tags, list):
        tags = [tags]

    if not isinstance(keywords, list):
        keywords = [keywords]

    combined = tags + keywords

    clean_tags = sorted(
        {
            str(tag).strip()
            for tag in combined
            if tag is not None and str(tag).strip() != ""
        },
        key=lambda tag: tag.lower()
    )

    return clean_tags


def normalize_dataset_records(data, source_file):
    records = []

    # Shape:
    # {
    #   "hafler-pmdbs-sn-rnaseq-pfc": {...},
    #   "another-dataset": {...}
    # }
    if isinstance(data, dict) and not any(
        key in data for key in ["name", "title", "description", "collection", "releases", "buckets"]
    ):
        for key, value in data.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("id", key)
                record.setdefault("name", key)
                record["_source_file"] = str(source_file)
                records.append(record)

    return records

def get_latest_release_info(dataset):
    releases = dataset.get("releases", {})

    if not isinstance(releases, dict) or not releases:
        return {
            "release": "TBD",
            "dataset_version": dataset.get("dataset_version", "TBD"),
            "cde_version": dataset.get("cde_version", "TBD"),
        }

    release_versions = sorted(releases.keys(), key=version_key, reverse=True)
    latest_release = release_versions[0]
    latest_info = releases.get(latest_release, {})

    if not isinstance(latest_info, dict):
        latest_info = {}

    return {
        "release": latest_release,
        "dataset_version": latest_info.get("dataset_version", dataset.get("dataset_version", "TBD")),
        "cde_version": latest_info.get("cde_version", dataset.get("cde_version", "TBD")),
    }


# ----------------------------
# Load dataset JSON
# ----------------------------

datasets = []

if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Could not find dataset folder: {DATASET_DIR}")

for file in DATASET_DIR.rglob("*.json"):
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        datasets.extend(normalize_dataset_records(data, file))

    except Exception as error:
        print(f"Skipping {file}: {error}")

if not datasets:
    raise ValueError("No dataset records found.")

# Deduplicate by dataset ID
unique_datasets = {}

for dataset in datasets:
    dataset_id = get_dataset_id(dataset)

    if dataset_id not in unique_datasets:
        unique_datasets[dataset_id] = dataset

datasets = sorted(
    unique_datasets.values(),
    key=lambda d: str(get_dataset_id(d)).lower()
)

all_tags = sorted(
    {
        tag
        for dataset in datasets
        for tag in get_tags(dataset)
    },
    key=lambda tag: tag.lower()
)

print(f"Loaded {len(datasets)} unique datasets")
print(f"Loaded {len(all_tags)} unique tags")


# ----------------------------
# Generate datasets.md
# ----------------------------

tag_options = ['<option value="">All tags</option>']

for tag in all_tags:
    tag_options.append(
        f'<option value="{esc(tag.lower())}">{esc(tag)}</option>'
    )

lines = [
    "# CRN Cloud Datasets",
    "",
    "Use this table to find dataset records, review versioning, and locate related release information.",
    "",
    '<div class="dataset-filters">',
    '  <input id="datasetSearch" class="dataset-search" type="text" placeholder="Filter by dataset, title, collection, tag, release, DOI, CDE version, or bucket path...">',
    '  <select id="tagFilter" class="tag-filter">',
    *tag_options,
    "  </select>",
    "</div>",
    "",
    '<p id="datasetCount" class="dataset-count"></p>',
    "",
    "<style>",
    ".md-grid {",
    "  max-width: 75rem;",
    "}",
    ".dataset-filters {",
    "  display: flex;",
    "  gap: 0.75rem;",
    "  align-items: center;",
    "  margin: 1rem 0 0.5rem 0;",
    "}",
    ".dataset-search {",
    "  flex: 1;",
    "  padding: 0.75rem;",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.45rem;",
    "  font-size: 1rem;",
    "}",
    ".tag-filter {",
    "  min-width: 240px;",
    "  padding: 0.75rem;",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.45rem;",
    "  font-size: 1rem;",
    "  background: var(--md-default-bg-color);",
    "  color: var(--md-default-fg-color);",
    "}",
    "@media (max-width: 700px) {",
    "  .dataset-filters {",
    "    flex-direction: column;",
    "    align-items: stretch;",
    "  }",
    "  .tag-filter {",
    "    width: 100%;",
    "  }",
    "}",
    ".dataset-count {",
    "  margin: 0 0 1rem 0;",
    "  color: var(--md-default-fg-color--light);",
    "}",
    ".dataset-table-wrap {",
    "  width: 100%;",
    "  overflow-x: auto;",
    "}",
    ".dataset-table {",
    "  min-width: 50%;",
    "  width: 100%;",
    "  border-collapse: collapse;",
    "  font-size: 0.82rem;",
    "  table-layout: fixed;",
    "}",
    ".dataset-table th, .dataset-table td {",
    "  border-bottom: 1px solid var(--md-default-fg-color--lightest);",
    "  padding: 0.55rem;",
    "  text-align: left;",
    "  vertical-align: top;",
    "}",
    ".dataset-table th {",
    "  font-weight: 700;",
    "}",
    ".dataset-table th:nth-child(1), .dataset-table td:nth-child(1) {",
    "  width: 210px;",
    "}",
    ".dataset-table th:nth-child(2), .dataset-table td:nth-child(2) {",
    "  width: 280px;",
    "}",
    ".dataset-table th:nth-child(3), .dataset-table td:nth-child(3) {",
    "  width: 140px;",
    "}",
    ".dataset-table th:nth-child(4), .dataset-table td:nth-child(4) {",
    "  width: 120px;",
    "}",
    ".dataset-table th:nth-child(5), .dataset-table td:nth-child(5) {",
    "  width: 110px;",
    "}",
    ".dataset-table th:nth-child(6), .dataset-table td:nth-child(6) {",
    "  width: 100px;",
    "}",
    ".dataset-table th:nth-child(7), .dataset-table td:nth-child(7) {",
    "  width: 190px;",
    "}",
    ".dataset-table th:nth-child(8), .dataset-table td:nth-child(8) {",
    "  width: 80px;",
    "}",
    ".dataset-table td {",
    "  word-break: break-word;",
    "}",
    ".tag-pill {",
    "  display: inline-block;",
    "  padding: 0.12rem 0.4rem;",
    "  margin: 0.1rem 0.15rem 0.1rem 0;",
    "  border-radius: 999px;",
    "  background: var(--md-code-bg-color);",
    "  font-size: 0.72rem;",
    "  white-space: nowrap;",
    "}",
    ".dataset-toggle {",
    "  border: 1px solid var(--md-default-fg-color--lightest);",
    "  border-radius: 0.35rem;",
    "  padding: 0.3rem 0.55rem;",
    "  background: var(--md-default-bg-color);",
    "  cursor: pointer;",
    "}",
    ".dataset-toggle:hover {",
    "  border-color: var(--md-accent-fg-color);",
    "}",
    ".dataset-detail-row {",
    "  display: none;",
    "}",
    ".dataset-detail {",
    "  padding: 0.75rem;",
    "  border-left: 3px solid var(--md-accent-fg-color);",
    "  background: var(--md-code-bg-color);",
    "}",
    ".dataset-detail h4 {",
    "  margin-top: 0.75rem;",
    "  margin-bottom: 0.35rem;",
    "}",
    ".mini-table {",
    "  width: 100%;",
    "  border-collapse: collapse;",
    "  font-size: 0.82rem;",
    "}",
    ".mini-table th, .mini-table td {",
    "  border-bottom: 1px solid var(--md-default-fg-color--lightest);",
    "  padding: 0.4rem;",
    "  text-align: left;",
    "  vertical-align: top;",
    "}",
    "</style>",
    "",
    '<div class="dataset-table-wrap">',
    '<table class="dataset-table" id="datasetTable">',
    "  <thead>",
    "    <tr>",
    "      <th>Dataset</th>",
    "      <th>Title</th>",
    "      <th>Collection</th>",
    "      <th>Current dataset version</th>",
    "      <th>Latest release</th>",
    "      <th>CDE version</th>",
    "      <th>Tags</th>",
    "      <th>Details</th>",
    "    </tr>",
    "  </thead>",
    "  <tbody>",
]


for index, dataset in enumerate(datasets):
    dataset_id = get_dataset_id(dataset)
    dataset_title = get_dataset_title(dataset)
    collection = get_collection(dataset)
    license_value = str(dataset.get("license", "TBD"))
    description = str(dataset.get("description", ""))
    keywords = dataset.get("keywords", [])
    buckets = dataset.get("buckets", {})
    releases = dataset.get("releases", {})
    dataset_doi = dataset.get("doi", "")

    if not isinstance(keywords, list):
        keywords = [keywords]

    keywords = [str(keyword) for keyword in keywords if keyword is not None]

    if not isinstance(buckets, dict):
        buckets = {}

    if not isinstance(releases, dict):
        releases = {}

    tags = get_tags(dataset)
    tags_text = ", ".join(tags) if tags else "NA"
    tags_search = "||".join(tag.lower() for tag in tags)

    tags_html = (
        " ".join(f'<span class="tag-pill">{esc(tag)}</span>' for tag in tags)
        if tags
        else "NA"
    )

    latest = get_latest_release_info(dataset)

    detail_id = f"dataset-detail-{safe_id(dataset_id)}-{index}"

    release_search = " ".join(
        [
            f"{release_key} {release_info.get('dataset_version', '')} {release_info.get('cde_version', '')}"
            for release_key, release_info in releases.items()
            if isinstance(release_info, dict)
        ]
    )

    search_text = " ".join([
        str(dataset_id),
        str(dataset_title),
        str(description),
        str(collection),
        str(license_value),
        str(dataset_doi),
        " ".join(tags),
        " ".join(keywords),
        " ".join(str(value) for value in buckets.values()),
        " ".join(str(key) for key in releases.keys()),
        release_search,
    ]).lower()

    lines.extend([
        f'    <tr class="dataset-row" data-detail="{esc(detail_id)}" data-search="{esc(search_text)}" data-tags="{esc(tags_search)}">',
        f"      <td><code>{esc(dataset_id)}</code></td>",
        f"      <td>{esc(dataset_title)}</td>",
        f"      <td>{esc(collection)}</td>",
        f"      <td>{esc(latest['dataset_version'])}</td>",
        f"      <td>{esc(latest['release'])}</td>",
        f"      <td>{esc(latest['cde_version'])}</td>",
        f"      <td>{tags_html}</td>",
        f'      <td><button class="dataset-toggle" data-target="{esc(detail_id)}">View</button></td>',
        "    </tr>",
        f'    <tr id="{esc(detail_id)}" class="dataset-detail-row">',
        '      <td colspan="8">',
        '        <div class="dataset-detail">',
        f"          <h3>{esc(dataset_title)}</h3>",
        f"          <p><strong>Dataset ID:</strong> <code>{esc(dataset_id)}</code></p>",
        f"          <p><strong>Description:</strong> {esc(description) if description else 'TBD'}</p>",
        f"          <p><strong>Collection:</strong> {esc(collection)}</p>",
        f"          <p><strong>License:</strong> {esc(license_value)}</p>",
        f"          <p><strong>DOI:</strong> {doi_link(dataset_doi)}</p>",
        f"          <p><strong>Tags:</strong> {esc(tags_text)}</p>",
        f"          <p><strong>Keywords:</strong> {esc(', '.join(keywords)) if keywords else 'TBD'}</p>",
        "          <h4>Release history</h4>",
        '          <table class="mini-table">',
        "            <thead>",
        "              <tr>",
        "                <th>Release</th>",
        "                <th>Dataset version</th>",
        "                <th>CDE version</th>",
        "              </tr>",
        "            </thead>",
        "            <tbody>",
    ])

    if releases:
        for release_version in sorted(releases.keys(), key=version_key, reverse=True):
            release_info = releases.get(release_version, {})

            if not isinstance(release_info, dict):
                release_info = {}

            lines.extend([
                "              <tr>",
                f"                <td>{esc(release_version)}</td>",
                f"                <td>{esc(release_info.get('dataset_version', 'TBD'))}</td>",
                f"                <td>{esc(release_info.get('cde_version', 'TBD'))}</td>",
                "              </tr>",
            ])
    else:
        lines.extend([
            "              <tr>",
            '                <td colspan="3">No release history listed.</td>',
            "              </tr>",
        ])

    lines.extend([
        "            </tbody>",
        "          </table>",
        "          <h4>Bucket paths</h4>",
        '          <table class="mini-table">',
        "            <thead>",
        "              <tr>",
        "                <th>Environment</th>",
        "                <th>Bucket path</th>",
        "              </tr>",
        "            </thead>",
        "            <tbody>",
    ])

    if buckets:
        for environment, bucket_path in buckets.items():
            lines.extend([
                "              <tr>",
                f"                <td>{esc(environment)}</td>",
                f"                <td><code>{esc(bucket_path)}</code></td>",
                "              </tr>",
            ])
    else:
        lines.extend([
            "              <tr>",
            '                <td colspan="2">No bucket paths listed.</td>',
            "              </tr>",
        ])

    lines.extend([
        "            </tbody>",
        "          </table>",
        "        </div>",
        "      </td>",
        "    </tr>",
    ])

lines.extend([
    "  </tbody>",
    "</table>",
    "</div>",
    "",
])

markdown_text = "\n".join(lines)

OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
OUT_FILE.write_text(markdown_text, encoding="utf-8")


# ----------------------------
# Generate JavaScript separately
# ----------------------------

js_text = """
function initializeDatasetPage() {
  const table = document.getElementById("datasetTable");

  if (!table) {
    return;
  }

  if (table.getAttribute("data-initialized") === "true") {
    return;
  }

  table.setAttribute("data-initialized", "true");

  const searchInput = document.getElementById("datasetSearch");
  const tagFilter = document.getElementById("tagFilter");
  const datasetCount = document.getElementById("datasetCount");
  const rows = Array.from(document.querySelectorAll(".dataset-row"));
  const buttons = Array.from(document.querySelectorAll(".dataset-toggle"));

  if (!rows.length) {
    return;
  }

  function updateCount(visibleCount) {
    if (datasetCount) {
      datasetCount.textContent = visibleCount + " of " + rows.length + " datasets shown";
    }
  }

  function closeDetailRow(row) {
    const detailId = row.getAttribute("data-detail");
    const detailRow = document.getElementById(detailId);
    const button = row.querySelector(".dataset-toggle");

    if (detailRow) {
      detailRow.style.display = "none";
    }

    if (button) {
      button.textContent = "View";
    }
  }

  function applyFilters() {
    const query = searchInput ? searchInput.value.toLowerCase().trim() : "";
    const selectedTag = tagFilter ? tagFilter.value.toLowerCase().trim() : "";

    let visibleCount = 0;

    rows.forEach(function (row) {
      const text = row.getAttribute("data-search") || "";
      const tags = row.getAttribute("data-tags") || "";

      const tagList = tags
        .split("||")
        .map(function (tag) {
          return tag.trim();
        })
        .filter(Boolean);

      const matchesText = query === "" || text.includes(query);
      const matchesTag = selectedTag === "" || tagList.includes(selectedTag);

      const isVisible = matchesText && matchesTag;

      row.style.display = isVisible ? "table-row" : "none";

      if (!isVisible) {
        closeDetailRow(row);
      }

      if (isVisible) {
        visibleCount += 1;
      }
    });

    updateCount(visibleCount);
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () {
      const targetId = button.getAttribute("data-target");
      const detailRow = document.getElementById(targetId);

      if (!detailRow) {
        return;
      }

      const isOpen = detailRow.style.display === "table-row";
      detailRow.style.display = isOpen ? "none" : "table-row";
      button.textContent = isOpen ? "View" : "Hide";
    });
  });

  if (searchInput) {
    searchInput.addEventListener("input", applyFilters);
  }

  if (tagFilter) {
    tagFilter.addEventListener("change", applyFilters);
  }

  updateCount(rows.length);
}

if (typeof document$ !== "undefined") {
  document$.subscribe(function () {
    initializeDatasetPage();
  });
} else {
  document.addEventListener("DOMContentLoaded", initializeDatasetPage);
}
"""

JS_FILE.parent.mkdir(parents=True, exist_ok=True)
JS_FILE.write_text(js_text.strip() + "\n", encoding="utf-8")

print(f"Wrote: {OUT_FILE}")
print(f"Wrote: {JS_FILE}")

Learning Lab root: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab
Dataset JSON folder: /Users/amaraalexander/Documents/GitHub/cloud-datasets
Output Markdown: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/datasets.md
Output JavaScript: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/dataset-filter.js
Loaded 62 unique datasets
Loaded 48 unique tags
Wrote: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/rosetta-stone/datasets.md
Wrote: /Users/amaraalexander/Documents/GitHub/asap-crn-learning-lab/docs/javascripts/dataset-filter.js


In [ ]:
def normalize_dataset_records(data, source_file):
    records = []

    # Shape:
    # {
    #   "hafler-pmdbs-sn-rnaseq-pfc": {...},
    #   "another-dataset": {...}
    # }
    if isinstance(data, dict) and not any(
        key in data for key in ["name", "title", "description", "collection", "releases", "buckets"]
    ):
        for key, value in data.items():
            if isinstance(value, dict):
                record = dict(value)
                record.setdefault("id", key)
                record.setdefault("name", key)
                record["_source_file"] = str(source_file)
                records.append(record)

    return records